# 1- Importações

In [1]:
import sys
print(sys.executable)

c:\Projects\Tech-Challenge-Fase-01\churn-prediction\.venv\Scripts\python.exe


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

# 2- Carregamento dos dados

In [3]:
path = "../data/raw/Telco_customer_churn.xlsx"
df = pd.read_excel(path)

# target
target_col = "Churn Value"

# remover leakage + irrelevantes
drop_cols = [
    "CustomerID", "Count", "Lat Long",
    "Churn Label", "Churn Score", "Churn Reason",
    "City", "State", "Zip Code"
]

df = df.drop(columns=drop_cols, errors="ignore")

# separar X e y
X = df.drop(columns=[target_col])
y = df[target_col]

# 3- Identificação de tipos de variáveis

In [4]:
num_cols = X.select_dtypes(include=np.number).columns
cat_cols = X.select_dtypes(include=["object", "string", "category"]).columns

# evitar erro do encoder
X[cat_cols] = X[cat_cols].astype("string")
X[cat_cols] = X[cat_cols].fillna("missing")

# 4- Pipeline de pré-processamento
 

In [5]:
# identificar colunas categóricas
cat_cols = X.select_dtypes(include=["object", "string"]).columns

# converter tudo para string
X[cat_cols] = X[cat_cols].astype("string")

# limpar valores problemáticos
X[cat_cols] = X[cat_cols].replace(" ", np.nan)

# preencher missing values
X[cat_cols] = X[cat_cols].fillna("missing")

# SimpleImputer:
#   Substitui valores faltantes pela mediana
#   Mais robusto que média (menos sensível a outliers)

# StandardScaler:
#   Centraliza (média = 0)
#   Escala (desvio padrão = 1)

# Numéricas
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# most_frequent:
#   Substitui missing pelo valor mais comum

# OneHotEncoder:
#   Converte categorias em colunas binárias
#   handle_unknown="ignore" evita erro em produção

# Categóricas
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Aplica transformações diferentes por tipo de coluna
# Garante pipeline único e consistente
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

# 5- Modelos Baseline

In [6]:
# Sempre prevê a classe mais frequente
# Não aprende nada

# linha de base mínima
# valida se seu modelo realmente aprende algo ou não

# DummyClassifier
dummy_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", DummyClassifier(strategy="most_frequent"))
])


# Modelo linear probabilístico
# Estima probabilidade de churn

# Logistic Regression
log_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=1000,class_weight="balanced"))
])

# 6- Métricas

In [7]:
from sklearn.metrics import make_scorer, precision_score

scoring = {
    "f1": "f1", # F1-score → equilíbrio entre precisão e recall
    "roc_auc": "roc_auc", # ROC-AUC → capacidade geral de separação
    "precision": make_scorer(precision_score, zero_division=0), # Precision → evita falsos positivos
    "recall": "recall" # Recall → captura churn (CRÍTICO)
}

# 7- Validação cruzada estratificada

In [8]:
# Divide dados em 5 partes (n_splits=5)
# Mantém proporção de churn em cada split

# Muito importante para:
# viés
# overfitting
# avaliação instável

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 8- Treinamento (Dummy)

In [9]:
# Treina e valida automaticamente
# Retorna métricas para cada fold

dummy_results = cross_validate(
    dummy_model,
    X,
    y,
    cv=cv,
    scoring=scoring,
    return_train_score=False
)
#print(dummy_results)

# 9- Treinamento — Logistic Regression

In [10]:
log_results = cross_validate(
    log_model,
    X,
    y,
    cv=cv,
    scoring=scoring,
    return_train_score=False
)

#print(log_results)

# 10- Consolidação dos resultados

In [11]:
results_df = pd.DataFrame({
    "Dummy": [np.mean(dummy_results[f"test_{m}"]) for m in scoring],
    "Logistic": [np.mean(log_results[f"test_{m}"]) for m in scoring]
}, index=scoring.keys())

print(results_df)

           Dummy  Logistic
f1           0.0  0.644705
roc_auc      0.5  0.856373
precision    0.0  0.549976
recall       0.0  0.779037


# MLflow ------------------------------------------------------------------

# 1- Dataset version

In [12]:
import hashlib
import pandas as pd

def get_dataset_version(df):
    return hashlib.md5(
        pd.util.hash_pandas_object(df, index=True).values
    ).hexdigest()

dataset_version = get_dataset_version(df)

# 2- Setup MLflow

In [13]:
import mlflow
print("MLflow OK 🚀")

MLflow OK 🚀


In [14]:
import mlflow
import mlflow.sklearn

mlflow.set_experiment("churn-baseline")

2026/04/12 13:44:19 INFO mlflow.tracking.fluent: Experiment with name 'churn-baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///C:/Projects/Tech-Challenge-Fase-01/churn-prediction/notebooks/mlruns/1', experiment_id='1', lifecycle_stage='active', name='churn-baseline', tags={}>

# 3- Registrar experimento (Logistic)

In [15]:
from sklearn.model_selection import cross_validate

if mlflow.active_run():
    mlflow.end_run()

mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("churn-baseline")

with mlflow.start_run(run_name="LogisticRegression"):

    results = cross_validate(
        log_model,
        X,
        y,
        cv=5,
        scoring={
            "f1": "f1",
            "roc_auc": "roc_auc",
            "precision": "precision",
            "recall": "recall"
        }
    )

    # métricas
    mlflow.log_metric("f1", results["test_f1"].mean())
    mlflow.log_metric("roc_auc", results["test_roc_auc"].mean())
    mlflow.log_metric("precision", results["test_precision"].mean())
    mlflow.log_metric("recall", results["test_recall"].mean())

    # parâmetros
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("class_weight", "balanced")

    # dataset
    mlflow.log_param("dataset_version", dataset_version)
    mlflow.log_param("n_rows", len(df))
    mlflow.log_param("n_features", X.shape[1])

    # features usadas
    mlflow.log_param("features", list(X.columns))

    # treinar e salvar modelo
    log_model.fit(X, y)
    mlflow.sklearn.log_model(log_model, "model")


2026/04/12 13:44:24 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\alexi\AppData\Local\Temp\tmp8x0d1q6i\model\model.pkl, flavor: sklearn), fall back to return ['scikit-learn==1.8.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback.
2026/04/12 13:44:24 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


# 4- Resultados

In [16]:
results_df = pd.DataFrame({
    "f1": results["test_f1"],
    "roc_auc": results["test_roc_auc"],
    "precision": results["test_precision"],
    "recall": results["test_recall"]
})

results_df.to_csv("results.csv", index=False)
mlflow.log_artifact("results.csv")